In [62]:
!pip install transformers peft bitsandbytes accelerate datasets tqdm huggingface_hub

In [63]:
import json
import torch
from typing import Dict, List, Any
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
from huggingface_hub import login
from training_config import get_config, print_config, TrainingConfig

In [64]:
# Custom data collator
class EntityRecognitionCollator(DataCollatorForLanguageModeling):
    """Custom collator for entity recognition training."""

    def __call__(self, features):
        # Get the maximum length in this batch
        max_length = max(len(feature['input_ids']) for feature in features)

        # Pad all sequences to the same length
        batch = {}
        for key in ['input_ids', 'attention_mask', 'labels']:
            batch[key] = []
            for feature in features:
                sequence = feature[key]
                # Pad with appropriate values
                if key == 'labels':
                    # Use -100 for padding in labels (ignored by loss function)
                    padded = sequence + [-100] * (max_length - len(sequence))
                else:
                    # Use pad_token_id for input_ids and 0 for attention_mask
                    if key == 'input_ids':
                        pad_value = self.tokenizer.pad_token_id
                    else:
                        pad_value = 0
                    padded = sequence + [pad_value] * (max_length - len(sequence))
                batch[key].append(padded)

        # Convert to tensors
        for key in batch:
            batch[key] = torch.tensor(batch[key])

        return batch

In [65]:
# Change this line to switch models
MODEL_NAME = "qwen-1.5b"  # Options: "qwen-1.5b", "qwen-0.5b"

# Get configuration
config = get_config(
    model_name="qwen-1.5b",
    train_data_path="./training_data_entity_recognition.json",  # Root directory
    prompt_path="./entity_recognition_prompt.txt",              # Root directory
    output_dir="./entity_recognition_model_continued"           # Root directory
)
print_config(config)

Training Configuration:
  Model: qwen-1.5b (Qwen/Qwen2.5-1.5B-Instruct)
  Max Length: 1024
  Batch Size: 4
  Learning Rate: 0.0002
  Epochs: 3
  LoRA r: 16
  LoRA alpha: 32
  Output Dir: ./entity_recognition_model_continued


In [66]:
# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check HuggingFace login
try:
    from huggingface_hub import whoami
    username = whoami()
    print(f"Logged in to HuggingFace as: {username}")
except Exception:
    print("Warning: Not logged in to HuggingFace Hub")
    print("To push model after training, run: huggingface-cli login")

GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.5 GB
Logged in to HuggingFace as: {'type': 'user', 'id': '686e2f4c8cc9f98517dc3543', 'name': 'ssuki', 'fullname': 'Sukruthi Santosh', 'isPro': False, 'avatarUrl': '/avatars/7b5fc575d5e1611a60c25dbf95870267.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'bi-intent-token', 'role': 'fineGrained', 'createdAt': '2025-08-17T18:11:53.275Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '686e2f4c8cc9f98517dc3543', 'type': 'user', 'name': 'ssuki'}, 'permissions': ['repo.content.read', 'repo.write']}]}}}}


In [67]:
def load_training_data(config: TrainingConfig) -> List[Dict[str, Any]]:
    """Load training data from JSON file."""
    with open("./examples_101-200.json", 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Use all data in the file (don't extract subset)
    print(f"Loaded {len(data)} training examples from {config.train_data_path}")
    return data

def load_prompt_template(config: TrainingConfig) -> str:
    """Load the entity recognition prompt template."""
    with open(config.prompt_path, 'r', encoding='utf-8') as f:
        return f.read().strip()

def split_data(data: List[Dict[str, Any]], train_ratio: float = 0.8):
    """Split data into train/validation sets."""
    train_size = int(train_ratio * len(data))
    train_data = data[:train_size]
    val_data = data[train_size:]

    print(f"Train examples: {len(train_data)}")
    print(f"Validation examples: {len(val_data)}")

    return train_data, val_data

In [68]:
class EntityRecognitionDataset(Dataset):
    """Dataset for entity recognition training."""

    def __init__(self, data: List[Dict[str, Any]], tokenizer, prompt_template: str, max_length: int = 1024):
        self.data = data
        self.tokenizer = tokenizer
        self.prompt_template = prompt_template
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]

        # Create input prompt
        input_text = self.prompt_template.format(question=example['input'])

        # Create expected output
        output_text = json.dumps(example['output'], ensure_ascii=False, separators=(',', ':'))

        # Combine input and output
        full_text = input_text + output_text

        # Tokenize WITHOUT padding
        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding=False,  # No padding here
            return_tensors=None
        )

        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': encoding['input_ids'].copy()
        }

In [69]:
def load_model_and_tokenizer(config: TrainingConfig):
    """Load the pre-trained model from Hugging Face."""
    print("Loading pre-trained model from Hugging Face...")

    # Load your trained model
    model_name = "ssuki/qwen-1.5b-entity-recognition"  # Your model name

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

    # Prepare for training
    model = prepare_model_for_kbit_training(model)

    print(f"✅ Loaded pre-trained model: {model_name}")
    return model, tokenizer

def setup_lora(model, config: TrainingConfig):
    """Setup LoRA for efficient fine-tuning."""
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()

    print("LoRA applied successfully")
    model.print_trainable_parameters()

    return model, lora_config

In [70]:
# Load data
data = load_training_data(config)
train_data, val_data = split_data(data)

# Load prompt template
prompt_template = load_prompt_template(config)

# Load model and tokenizer
model, tokenizer = load_model_and_tokenizer(config)

# Setup LoRA
model, lora_config = setup_lora(model, config)

Loaded 100 training examples from ./training_data_entity_recognition.json
Train examples: 80
Validation examples: 20
Loading pre-trained model from Hugging Face...
✅ Loaded pre-trained model: ssuki/qwen-1.5b-entity-recognition


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


LoRA applied successfully
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [71]:
def create_datasets(train_data, val_data, tokenizer, prompt_template, config: TrainingConfig):
    """Create training and validation datasets."""
    train_dataset = EntityRecognitionDataset(train_data, tokenizer, prompt_template, config.model.max_length)
    val_dataset = EntityRecognitionDataset(val_data, tokenizer, prompt_template, config.model.max_length)
    return train_dataset, val_dataset

def train_model(model, tokenizer, train_dataset, val_dataset, config: TrainingConfig):
    """Train the model."""
    print("Starting training...")

    training_args = TrainingArguments(
        output_dir=config.output_dir,
        num_train_epochs=1,  # Reduced from 3 to 1 since continuing training
        per_device_train_batch_size=config.model.batch_size,
        per_device_eval_batch_size=config.model.batch_size,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,  # Reduced learning rate for fine-tuning
        warmup_steps=10,     # Reduced warmup
        logging_steps=config.logging_steps,
        save_steps=config.save_steps,
        eval_steps=config.eval_steps,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
        report_to=None,
    )

    # Use custom collator
    data_collator = EntityRecognitionCollator(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(config.output_dir)

    print(f"Training completed! Model saved to {config.output_dir}")
    return trainer

In [72]:
# Create datasets
train_dataset, val_dataset = create_datasets(train_data, val_data, tokenizer, prompt_template, config)

# Train model
trainer = train_model(model, tokenizer, train_dataset, val_dataset, config)

Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


Training completed! Model saved to ./entity_recognition_model_continued


In [73]:
def test_model(model, tokenizer, test_questions: List[str], config: TrainingConfig):
    """Test the trained model on sample questions."""
    print("\nTesting model on sample questions...")

    # Temporarily disable gradient checkpointing for faster generation
    model.gradient_checkpointing_disable()

    prompt_template = load_prompt_template(config)

    for question in test_questions:
        print(f"\nQuestion: {question}")

        input_text = prompt_template.format(question=question)
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=config.model.max_length)

        # Move inputs to the same device as the model
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = generated_text[len(input_text):].strip()

        print(f"Response: {response}")

        try:
            parsed = json.loads(response)
            print(f"Valid JSON: {json.dumps(parsed, indent=2)}")
        except json.JSONDecodeError:
            print("Invalid JSON format")

    # Re-enable gradient checkpointing
    model.gradient_checkpointing_enable()

In [74]:
# Test model
test_questions = [
    "How many heads of the publishers are older than 56?",
    "What is the average revenue of departments?",
    "List the names of publishers created in California"
]

test_model(model, tokenizer, test_questions, config)


Testing model on sample questions...

Question: How many heads of the publishers are older than 56?
Response: ```json
{
  "dimensions": ["publisher", "publishers", "age", "older than 56"],
  "measures": ["heads"],
  "calculations": ["count"],
  "filters": ["older than 56"],
  "time_references": []
}Human Resources - Employee Performance Analysis

### Dimensions
- **Employee Name**
- **Department**
- **Job Title**
- **Performance Rating**

### Measures
- **Average Salary**
- **Total Hours Worked**
- **Number of Projects Completed**

### Calculations
- **Sum**
- **Count**

### Filters
- **
Invalid JSON format

Question: What is the average revenue of departments?
Response: ```json
{
  "dimensions": ["department"],
  "measures": ["revenue"],
  "calculations": ["average"],
  "filters": [],
  "time_references": []
}Human Resources - Department Manager - Salary Increase - Last Year - Average Salary

## Response:

```json
{
  "dimensions": ["department", "manager", "salary increase", "last y

In [75]:
def push_to_hub(model, tokenizer, config: TrainingConfig):
    """Push the trained model to HuggingFace Hub."""
    print("\nPushing model to HuggingFace Hub...")

    repo_name = f"ssuki/{config.model.name}-entity-recognition"

    try:
        model.push_to_hub(repo_name, private=False)
        tokenizer.push_to_hub(repo_name, private=False)
        print(f"Successfully pushed to: https://huggingface.co/{repo_name}")
    except Exception as e:
        print(f"Error pushing to Hub: {e}")
        print("Make sure you're logged in: huggingface-cli login")

# Push to HuggingFace Hub
push_to_hub(model, tokenizer, config)


Pushing model to HuggingFace Hub...


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...p2dx7z8_p/adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmprhz8ox7n/tokenizer.json       : 100%|##########| 11.4MB / 11.4MB            

Successfully pushed to: https://huggingface.co/ssuki/qwen-1.5b-entity-recognition
